# Data Cleaning

This notebook performs end-to-end data cleaning, validation, outlier treatment, and feature engineering with explanations.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Dataset

In [2]:
df = pd.read_csv('Dataset.csv')
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 3. Dataset Overview

In [15]:
print('Shape:', df.shape)
display(df.info())
display(df.describe())

Shape: (1200, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       1200 non-null   object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
 14  Year             1200 

None

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice,Year,Month,Day
count,1200,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.643183,2023.767500,5.995000,15.969167
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000,2023.000000,1.000000,1.000000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000,2023.000000,3.000000,8.000000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000,2024.000000,6.000000,16.000000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000,2024.000000,9.000000,24.000000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3330.407500,2025.000000,12.000000,31.000000
std,NaN,1.407557,197.177146,2.281983,818.937858,0.750942,3.344293,8.762450


## 4. Check Missing Values

In [4]:
missing = df.isnull().sum()
missing[missing > 0]

CouponCode    309
dtype: int64

### Handle Missing Values

In [16]:
if 'CouponCode' in df.columns:
    df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')

## 5. Remove Duplicate Records

In [17]:
duplicates = df.duplicated().sum()
print('Duplicate Rows:', duplicates)
df = df.drop_duplicates()

Duplicate Rows: 0


## 6. Standardize Column Names

In [7]:
df.columns = df.columns.str.strip().str.replace(' ', '_')
df.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

## 7. Data Type Conversion

In [8]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df.dtypes

OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

## 8. Clean Text Columns

In [18]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

## 9. Outlier Detection and Treatment (IQR Method)

In [19]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print('Outlier treatment completed.')

Outlier treatment completed.


## 10. Data Validation

In [11]:
if 'Quantity' in df.columns:
    df = df[df['Quantity'] > 0]

print(df.shape)

(1200, 14)


## 11. Feature Engineering

In [12]:
if 'Date' in df.columns:
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day

df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,Year,Month,Day
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,2023,1,4
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,2024,8,23
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,2024,2,27
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,2023,10,15
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,2025,5,8


## 12. Final Quality Check

In [13]:
print(df.isnull().sum())
print(df.shape)

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
Year               0
Month              0
Day                0
dtype: int64
(1200, 17)


## 13. Save Cleaned Dataset

In [14]:
df.to_csv('cleaned_dataset.csv', index=False)
print('Dataset saved successfully.')

Dataset saved successfully.


## Conclusion

The dataset has been cleaned through:
- Missing value treatment
- Duplicate removal
- Data type correction
- Text cleaning
- Outlier handling
- Data validation
- Feature engineering